In [2]:
import time
from tabulate import tabulate

In [3]:
# Running the file containing all my implementations of NTT, Pt-NTT and K-NTT

%run allImplementations.ipynb

# Time measurement

## Generating a number of random polynomials in Z_q[x]/(x^n-a)

In [4]:
# input is the number m of random polynomials to be generated, n the degree of the polynomials and q the prime
# return a numpy array that contains m pairs of randomly generated polynomials of length n
def randPolyArray(m,n,q):
    arr = np.zeros(shape=(m,2,n),dtype=int)

    for instance in arr:
        instance[0] = genRandPoly(n,q)
        instance[1] = genRandPoly(n,q)

    return arr

## Function for measuring the time

In [5]:
# time.process_time_ns() → int

# Return the value (in nanoseconds) of the sum of the system and user CPU time of the current process.
# It does not include time elapsed during sleep. It is process-wide by definition. 
# The reference point of the returned value is undefined, so that only the difference between the results of two 
# calls is valid.

In [7]:
# writing a function that should take as an input all multiplication methods that are to be compared in the form of
# a dictionary,
# the number m of polynomial multiplications, n and q,
# all necessary parameters for the multiplication methods that are compared in **kwargs

# the necessary **kwargs are:
    # - a for schoolbook multiplication
    # - w,full_k for generalized NTT
    # - w,alpha,split_k for Pt-NTT
    # - w,alpha,split_k for K-NTT

# the arrays for the NTT and the INTT are created inside the timing function

# if both generalized NTT and either Pt-NTT or K-NTT are computed, then split_k = full_k * (2^alpha) such that 
# a = w^(n*full_k) = w^(n/(2^alpha)*split_k)

# output is a dictionary with the times

def timing(to_time,m,n,q,**kwargs):
    # the array containing the m pairs of random polynomials and a hard copy of it for every multiplication method
    originalArr = randPolyArray(m,n,q)
    copiedArr = {name:np.copy(originalArr) for name in to_time}

    # an array to store the results
    resultArr = {name:np.zeros(shape=(m,n),dtype=int) for name in to_time}
    
    # the dictionary to store the resulting times
    times = {} 

    for method in to_time:
        # doing the precomputations and creating the lambda function which will be used to measure the time
        if method == "Schoolbook Multiplication":
            function = lambda f,g: to_time[method](f,g,n,q,kwargs.get('a'))
            
        elif method == "Generalized NTT":
            # computing arrays and the parameter n_inv
            full_NTTarray,full_INTTarray = compute_arrays(kwargs.get('w'),n,q,kwargs.get('full_k'))
            n_inv = multInverse(n,q)
            
            function = lambda f,g: to_time[method](f,g,n,q,n_inv,full_NTTarray,full_INTTarray)

        elif method == "Pt-NTT":
            # unpacking kwargs
            w = kwargs.get('w')
            split_k = kwargs.get('split_k')
            alpha = kwargs.get('alpha')

            # computing parameters and arrays
            columns = n//(2**alpha)
            columns_inv = multInverse(columns,q)
            a = fast2Power(w,columns*split_k,q)
            split_NTTarray,split_INTTarray = compute_arrays(w,columns,q,split_k)
            
            function = lambda f,g: to_time[method](f,g,n,q,a,alpha,columns_inv,split_NTTarray,split_INTTarray)

        elif method == "K-NTT":
            # unpacking kwargs
            w = kwargs.get('w')
            split_k = kwargs.get('split_k')
            alpha = kwargs.get('alpha')

            # computing parameters and arrays
            columns = n//(2**alpha)
            columns_inv = multInverse(columns,q)
            a = fast2Power(w,columns*split_k,q)
            split_NTTarray,split_INTTarray = compute_arrays(w,columns,q,split_k)
            
            # precomputing the NTT of y
            y = np.zeros(columns,dtype=int)
            y[1]=1
            NTTy = NTT(y,columns,q,split_NTTarray)
            
            function = lambda f,g: to_time[method](f,g,n,q,a,alpha,columns_inv,split_NTTarray,split_INTTarray,NTTy)
        
        # time.process_time_ns() or time.perf_counter_ns()
        start_time = time.process_time_ns()
        
        for element in copiedArr[method]:
            element[0] = function(element[0],element[1])
        
        # same as in start_time
        end_time = time.process_time_ns()

        # computing results and adding to times dictionary
        total_time = end_time-start_time
        timePerMult = total_time / m

        times[method] = {"Total time in ns": total_time,"Time per multiplication in ns": timePerMult}

        # filling the resultArr
        for i in range(m):
            resultArr[method][i] = copiedArr[method][i][0]
    
    # checking if all methods that we used yield the same results
    check = all(np.array_equal(val, list(resultArr.values())[0]) for val in resultArr.values())
    print("All methods yield the same result: ",check)
    
    return times

## Testing

In [8]:
to_time = {
    "Schoolbook Multiplication": schoolMult,
    "Generalized NTT": polyMultbyNTT,
    "Pt-NTT": polyMultbyPtNTT,
    "K-NTT": polyMultbykNTT
}

In [9]:
# Kyber parameters
n = 256
q = 3329
# 3329 = 2^8 * 13

m = 10000

primeList = [2,13]
w = findGenerator(q,primeList)
print(w)


full_k = 7
a = fast2Power(w,n*full_k,q)
print(a)

alpha = 1
split_k = full_k*(2**alpha)
print(fast2Power(w,split_k*(n//(2**alpha)),q))

3
2764
2764


In [10]:
results = timing(to_time,m,n,q,a=a,w=w,full_k=full_k,alpha=alpha,split_k=split_k)

print(results)

All methods yield the same result:  True
{'Schoolbook Multiplication': {'Total time in ns': 637156250000, 'Time per multiplication in ns': 63715625.0}, 'Generalized NTT': {'Total time in ns': 39859375000, 'Time per multiplication in ns': 3985937.5}, 'Pt-NTT': {'Total time in ns': 50515625000, 'Time per multiplication in ns': 5051562.5}, 'K-NTT': {'Total time in ns': 41250000000, 'Time per multiplication in ns': 4125000.0}}


# Generating a table containing the results

In [11]:
# takes as input a dictionary which contains the results of the timing(...) function
# outputs a table (for printing in Python or Latex)
# tableformat can be changed to "latex", "latex_raw", "latex_booktabs" or "latex_longtable"

def print_results(results,tableformat="fancy_outline"):
    headers = ["Mult. method","Total time in ns","Time per mult. in ns","Methods ratio"]

    # reformatting the dictionary to a list
    table = []
    for element in results:
        table.append([element,results[element]["Total time in ns"],results[element]["Time per multiplication in ns"]])

    # computing "Slowest method divided by current method"
    total_times = []
    for row in table:
        total_times.append(row[1])
    largest_time = max(total_times)
    for row in table:
        row.append(row[1]/largest_time)
    print(tabulate(table,headers,tablefmt=tableformat))

In [12]:
print_results(results)

╒═══════════════════════════╤════════════════════╤════════════════════════╤═════════════════╕
│ Mult. method              │   Total time in ns │   Time per mult. in ns │   Methods ratio │
╞═══════════════════════════╪════════════════════╪════════════════════════╪═════════════════╡
│ Schoolbook Multiplication │       637156250000 │            6.37156e+07 │       1         │
│ Generalized NTT           │        39859375000 │            3.98594e+06 │       0.0625582 │
│ Pt-NTT                    │        50515625000 │            5.05156e+06 │       0.0792829 │
│ K-NTT                     │        41250000000 │            4.125e+06   │       0.0647408 │
╘═══════════════════════════╧════════════════════╧════════════════════════╧═════════════════╛


In [13]:
print_results(results,tableformat="latex")

\begin{tabular}{lrrr}
\hline
 Mult. method              &   Total time in ns &   Time per mult. in ns &   Methods ratio \\
\hline
 Schoolbook Multiplication &       637156250000 &            6.37156e+07 &       1         \\
 Generalized NTT           &        39859375000 &            3.98594e+06 &       0.0625582 \\
 Pt-NTT                    &        50515625000 &            5.05156e+06 &       0.0792829 \\
 K-NTT                     &        41250000000 &            4.125e+06   &       0.0647408 \\
\hline
\end{tabular}
